# 01. 토크나이저 비교 — 단일 뉴클레오타이드 · k-mer · BPE

**등급**: 🟢 Mac 로컬 (CPU) · **예상 시간**: 1~2분 · **네트워크**: 불필요

**목표**
1. 같은 서열에 세 가지 토크나이저를 적용해 **토큰 수 / 어휘 크기 / attention 비용**을 비교한다.
2. BPE 어휘 크기를 바꿔 **압축률**과 모티프 학습 여부를 본다.
3. 염기 1개 삽입에 대한 **프레임 이동 민감도**를 측정한다.
4. 겹침 k-mer에서 **마스킹 누출**이 얼마나 생기는지 계산한다.

> 이 노트북의 DNA는 **합성 데이터**(AT-rich 배경 + 가상의 모티프)다. 수치는 개념 시연이며 실제 게놈의 값이 아니다.

In [ ]:
import collections
import random

import matplotlib.pyplot as plt
import numpy as np
from tokenizers import Tokenizer, models, trainers

SEED = 0
rng = random.Random(SEED)
np.random.seed(SEED)

## 1. 합성 DNA 코퍼스
배경은 AT-rich 확률(A,T=0.3 / C,G=0.2)로 생성하고, 길이 8짜리 가상 모티프를 서열당 12개 무작위 위치에 심는다.
(단순화: 위치를 겹침 없이 고르지 않고 **덮어쓴다**. 그래서 모티프끼리 겹쳐 일부가 훼손되거나 이어붙은 모양(예: `GGGCGGGGGGCGGGG`)이 생길 수 있다.)

In [ ]:
BASES = "ACGT"
BG_P = [0.3, 0.2, 0.2, 0.3]
MOTIFS = ["TATAAAAG", "GGGCGGGG", "CACGTGAC", "TTGACATG"]  # 장난감 모티프 (실제 조절 모티프와 무관)
SEQ_LEN = 1000


def make_seq(length=SEQ_LEN, n_motifs=12):
    seq = rng.choices(BASES, weights=BG_P, k=length)
    for _ in range(n_motifs):
        m = rng.choice(MOTIFS)
        pos = rng.randrange(0, length - len(m))
        seq[pos:pos + len(m)] = list(m)
    return "".join(seq)


corpus = [make_seq() for _ in range(2000)]
print(len(corpus), "sequences x", SEQ_LEN, "nt")
print(corpus[0][:80])

## 2. 토크나이저 4종
- 단일 뉴클레오타이드: 문자 하나 = 토큰
- 겹침 k-mer: 1칸씩 이동하는 윈도 (DNABERT 방식)
- 비겹침 k-mer: k칸씩 이동 (마지막의 남는 조각은 단순화를 위해 버림)
- BPE: Hugging Face `tokenizers` 라이브러리로 코퍼스에서 학습

In [ ]:
K = 6


def tok_single(seq):
    return list(seq)


def tok_kmer_overlap(seq, k=K):
    return [seq[i:i + k] for i in range(len(seq) - k + 1)]


def tok_kmer_nonoverlap(seq, k=K):
    return [seq[i:i + k] for i in range(0, len(seq) - k + 1, k)]


def train_bpe(texts, vocab_size):
    tk = Tokenizer(models.BPE(unk_token="[UNK]"))
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]", "[PAD]", "[MASK]"],
        initial_alphabet=list(BASES),
        show_progress=False,
    )
    tk.train_from_iterator(texts, trainer)
    return tk


bpe512 = train_bpe(corpus, 512)


def tok_bpe(seq, tk=bpe512):
    return tk.encode(seq).tokens


print(tok_bpe(corpus[0])[:12])

## 3. 토큰 수 · 어휘 크기 · attention 비용
표준 attention 비용은 토큰 수의 제곱에 비례하므로, 단일 뉴클레오타이드(1000토큰)를 기준으로 상대 비용을 계산한다.

In [ ]:
def summarize(name, fn, vocab):
    n_tok = float(np.mean([len(fn(s)) for s in corpus[:200]]))
    return {"name": name, "vocab": vocab, "tokens": n_tok, "rel_attn": (n_tok / SEQ_LEN) ** 2}


rows = [
    summarize("single nt", tok_single, 4),
    summarize(f"k-mer overlap (k={K})", tok_kmer_overlap, 4 ** K),
    summarize(f"k-mer non-overlap (k={K})", tok_kmer_nonoverlap, 4 ** K),
    summarize("BPE (vocab=512)", tok_bpe, bpe512.get_vocab_size()),
]
print(f"{'tokenizer':26s}{'vocab':>8s}{'tokens/1000nt':>16s}{'rel. attn cost':>16s}")
for r in rows:
    print(f"{r['name']:26s}{r['vocab']:>8d}{r['tokens']:>16.1f}{r['rel_attn']:>16.3f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar([r["name"] for r in rows], [r["tokens"] for r in rows])
ax[0].set_title("tokens per 1000 nt")
ax[0].tick_params(axis="x", rotation=30)
ax[1].bar([r["name"] for r in rows], [r["rel_attn"] for r in rows])
ax[1].set_yscale("log")
ax[1].set_title("relative attention cost (log)")
ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

**생각해 볼 점**
- 겹침 k-mer는 어휘는 크지만 토큰 수가 서열 길이와 거의 같다. 비용 측면의 이득이 있는가?
- 비겹침 k-mer와 BPE는 비용이 줄어든다. 대신 무엇을 잃는가? (아래 4~5절)

## 4. BPE 어휘 크기 스윕 — 압축률과 모티프 학습
어휘 크기를 키우면 압축률은 어떻게 변하고, 심어 둔 8-mer 모티프가 **토큰 하나**로 학습되는 시점은 언제인가?

In [ ]:
sweep = {}
train_texts, test_texts = corpus[:1000], corpus[1000:1100]
for vs in [128, 256, 512, 1024, 2048]:
    tk = train_bpe(train_texts, vs)
    n_tok = float(np.mean([len(tk.encode(s).tokens) for s in test_texts]))
    vocab = tk.get_vocab()
    motif_len = {m: len(tk.encode(m).tokens) for m in MOTIFS}  # 모티프만 단독으로 인코딩했을 때 토큰 수 (1 = 단일 토큰)
    sweep[vs] = (n_tok, motif_len)
    longest = sorted(vocab, key=len, reverse=True)[:4]
    print(f"vocab={vs:5d}  tokens/1000nt={n_tok:7.1f}  compression={SEQ_LEN / n_tok:4.2f}x")
    print(f"           모티프별 토큰 수: {motif_len}")
    print(f"           가장 긴 토큰: {longest}")

fig, ax = plt.subplots(figsize=(4.5, 3))
ax.plot(list(sweep), [v[0] for v in sweep.values()], marker="o")
ax.set_xscale("log", base=2)
ax.set_xlabel("BPE vocab size")
ax.set_ylabel("tokens / 1000 nt")
plt.tight_layout()
plt.show()

**관찰 (seed=0 기준, 직접 확인할 것)**
- 어휘 크기를 키울수록 압축률이 올라간다.
- 모티프 4개 중 2개(`GGGCGGGG`, `CACGTGAC`)는 어휘 128에서 이미 **단일 토큰**이 되지만, 나머지 2개(`TATAAAAG`, `TTGACATG`)는 어휘를 2048까지 키워도
  단독 인코딩 시 3토큰으로 쪼개진다.
- 어휘가 커지면 가장 긴 토큰이 모티프 **주변의 무작위 서열까지 흡수**한다(예: `...CACGTGACT`, `...GGGCGGGGTT`).

즉 BPE의 토큰 경계는 **빈도**가 정하며 생물학적 단위(모티프)와 일치하리라는 보장이 없다.
해석 가능성이 중요하다면 토큰 경계와 기능 단위의 대응을 별도로 검증해야 한다. → 과제 1에서 모티프 길이·삽입 빈도를 바꿔 본다.

## 5. 프레임 이동(frame-shift) 민감도
서열 중간(위치 500)에 염기 1개를 삽입했다고 하자. **이상적인 토크나이저**라면 삽입 지점 이전의 토큰은 그대로, 이후의 토큰은 **같은 문자열이 좌표만 1칸 밀려** 나와야 한다.
실제로 그런지 보기 위해 각 토큰을 `(시작, 끝, 문자열)`로 기록하고, 삽입 후 좌표를 되돌려 **원래 토큰이 그대로 보존된 비율**을 잰다 (좌표를 되돌리는 것은 비교용 규칙일 뿐이며 토큰화 결과가 보존된다는 뜻이 아니다).

In [ ]:
INS_POS = 500


def spans_single(seq):
    return [(i, i + 1, c) for i, c in enumerate(seq)]


def spans_kmer(seq, k=K, stride=1):
    return [(i, i + k, seq[i:i + k]) for i in range(0, len(seq) - k + 1, stride)]


def spans_bpe(seq, tk=bpe512):
    enc = tk.encode(seq)
    return [(a, b, t) for (a, b), t in zip(enc.offsets, enc.tokens)]


def conserved_fraction(span_fn, n=100):
    fracs = []
    for s in corpus[:n]:
        s2 = s[:INS_POS] + "A" + s[INS_POS:]  # 위치 500에 'A' 삽입
        base = set(span_fn(s))
        mapped = set()
        for a, b, t in span_fn(s2):
            if b <= INS_POS:              # 삽입 지점 이전: 좌표 그대로
                mapped.add((a, b, t))
            elif a >= INS_POS + 1:        # 삽입 지점 이후: 좌표를 1칸 되돌림
                mapped.add((a - 1, b - 1, t))
            # 삽입된 염기를 포함하는 토큰은 제외
        fracs.append(len(base & mapped) / len(base))
    return float(np.mean(fracs))


results = {
    "single nt": conserved_fraction(spans_single),
    "k-mer overlap": conserved_fraction(lambda s: spans_kmer(s, stride=1)),
    "k-mer non-overlap": conserved_fraction(lambda s: spans_kmer(s, stride=K)),
    "BPE (512)": conserved_fraction(spans_bpe),
}
for name, v in results.items():
    print(f"{name:20s} 보존된 토큰 비율: {v:.3f}")

**해석 (seed=0 기준)**: 비겹침 k-mer는 삽입 지점 이후 전체가 새 프레임으로 바뀌므로 보존 비율이 대략 절반(삽입 위치가 중간이므로)이다.
단일 뉴클레오타이드(1.00), 겹침 k-mer(약 0.995), BPE(약 0.99)는 좌표를 되돌리면 거의 모두 보존된다. BPE는 병합 규칙이 국소적이어서 삽입 지점 주변의 토큰 몇 개만 달라진다.
→ indel(삽입·결실) 효과를 다루는 변이 모델이라면 이 성질이 중요하다.

## 6. 겹침 k-mer의 마스킹 누출
겹침 k-mer에서 토큰 하나를 마스킹해도, 그 토큰의 염기는 **이웃 토큰들**에 그대로 들어 있다.
마스킹하는 연속 토큰 수를 늘려 가며, 마스킹한 토큰의 염기 중 **가려지지 않은 비율**을 계산한다.

In [ ]:
def leaked_fraction(width, k=K, target=100, length=300):
    """target 토큰 주변 ±width개 토큰을 함께 마스킹했을 때, target의 염기 중 이웃(보이는) 토큰에 남은 비율."""
    spans = [(i, i + k) for i in range(length - k + 1)]
    t0, t1 = spans[target]
    covered = np.zeros(length, dtype=bool)
    for j, (a, b) in enumerate(spans):
        if abs(j - target) > width:       # 마스킹되지 않은 토큰만 '보임'
            covered[a:b] = True
    return float(covered[t0:t1].mean())


for w in range(0, 7):
    print(f"±{w} 토큰 마스킹 (총 {2 * w + 1}개) → 누출 비율 {leaked_fraction(w):.2f}")

**결론**: 한 토큰만 마스킹하면 누출이 100%다. 토큰의 염기를 모두 가리려면 **2k−1 = 11개의 연속 토큰**(±5)을 함께 마스킹해야 한다(위 출력에서 누출이 0이 되는 지점).
단, 이때 완전히 가려지는 것은 **가운데 토큰의 6개 염기**뿐이며, 마스킹한 구간의 양옆 가장자리 염기는 마스킹 밖 토큰에 일부 남는다.
겹침 k-mer로 MLM을 하려면 이런 연속 구간 마스킹이 필요하다는 점이 설계상 까다롭다. 비겹침 k-mer·BPE·단일 뉴클레오타이드는 이 문제가 없다(또는 덜하다).

## 7. (선택) 단백질에서 BPE는? — 합성 예
단백질은 어휘가 작고(20종) 서열이 짧아서 DNA와 사정이 다르다. 모티프가 없는 무작위 단백질 서열에서 BPE가 얻는 압축률을 본다.

In [ ]:
AA = "ACDEFGHIKLMNPQRSTVWY"
prot_rng = random.Random(1)
prot_corpus = ["".join(prot_rng.choices(AA, k=300)) for _ in range(2000)]


def train_bpe_prot(texts, vocab_size):
    tk = Tokenizer(models.BPE(unk_token="[UNK]"))
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size, special_tokens=["[UNK]", "[PAD]", "[MASK]"],
        initial_alphabet=list(AA), show_progress=False)
    tk.train_from_iterator(texts, trainer)
    return tk


for vs in [64, 256, 1024]:
    tk = train_bpe_prot(prot_corpus[:1500], vs)
    n_tok = np.mean([len(tk.encode(s).tokens) for s in prot_corpus[1500:1600]])
    print(f"vocab={vs:5d}  tokens/300aa={n_tok:6.1f}  compression={300 / n_tok:4.2f}x")

**관찰과 주의**: 모티프가 전혀 없는 무작위 서열인데도 어휘를 키우면 압축률이 올라간다. 우연히 자주 등장하는 짧은 조각이 병합되기 때문이다.
즉 **압축률 자체는 좋은 토크나이저의 증거가 아니다.** 실제 단백질에서 BPE가 이득인지는 실제 서열 코퍼스와 다운스트림 성능으로 검증해야 하며,
이 합성 결과를 일반화하면 안 된다.

## 정리
| 토크나이저 | 토큰 수 | 프레임 이동 | 마스킹 | 해상도 |
|---|---|---|---|---|
| 단일 nt | L | 안전 | 안전 | 최대 |
| 겹침 k-mer | ≈ L | 안전 | 누출 | 최대(토큰은 k-mer) |
| 비겹침 k-mer | L/k | 취약 | 안전 | 토큰 경계 k |
| BPE | 코퍼스 의존 | 국소적 | 안전 | 가변 |

**과제**: [../exercises.md](../exercises.md)의 과제 1.